<a href="https://colab.research.google.com/github/carlocamilloni/Structural-Bioinformatics/blob/main/Notebooks/t02_StructurePred.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Structure prediction of a complex: comparing tools

In this practical you work in a group on **a complex of several molecules whose experimental structure is not known**. You will use several structure prediction tools (homology modelling, AlphaFold2-multimer through ColabFold, AlphaFold3, and at least one more recent method) to make a hypothesis about the structure of the complex. Since there is no experimental structure to tell you who is right, **you have to build an argument**: How confident are the predictions? Do different tools agree? Is the interface plausible? What happens in a negative control? Which experiment would test your model?

The reference theory is that of Lecture 4:

*   https://github.com/carlocamilloni/Structural-Bioinformatics/blob/main/Notes/04_StructurePredictionDocking.pdf

**What you will do**

| step | where | what |
|:--|:--|:--|
| 1. The system | this notebook | get the sequences, look at the annotations and choose the constructs to model |
| 2. The components | this notebook | secondary structure prediction, models from the AlphaFold Database |
| 3. Homology modelling | SwissModel server | is there a template for the complex or for its parts? |
| 4. Complex prediction | ColabFold, AlphaFold Server, Boltz... | the same system, several tools, several models |
| 5. Analysis | this notebook | confidence, interface, agreement between tools, consensus interface |
| 6. Controls and hypotheses | both | negative control, in-silico mutations, other evidence, proposed experiments |
| 7. Report | `report_2_StructurePred.ipynb` | what you found and how sure you are |

The predictions are made *outside* this notebook (servers and GPUs), the **analysis is done here, in Python**, so that it is reproducible and the same for all the tools.

## Setup

We work in a folder of the virtual machine (**it disappears when the runtime is recycled**: download what you need, or set `use_drive = True` to work in your Google Drive). The cell can be run more than once. Then we install two small libraries for protein structures and 3D views.

In [ ]:
import os, sys
IN_COLAB = 'google.colab' in sys.modules

if 'BASE_DIR' not in globals():          # remember where we started, running this cell again does not move us further
    BASE_DIR = os.getcwd()

use_drive = False        # True: keep your files on Google Drive (Colab will ask your permission)
if use_drive:
    from google.colab import drive
    drive.mount('/content/drive')
    workdir = '/content/drive/MyDrive/Structural_Bioinformatics/T02'
else:
    workdir = os.path.join(BASE_DIR, 'T02')

os.makedirs(workdir, exist_ok=True)
%cd $workdir

# one folder for the results of each tool: this is where you will put the files you download from the servers
for tool in ['colabfold', 'af3', 'boltz', 'swissmodel']:
    os.makedirs(f'results/{tool}', exist_ok=True)

In [ ]:
%pip install -q biopython py3Dmol

The next cells define the functions that we use for the analysis. Their code is hidden (double click on a cell to read it): just run them, in order.

In [ ]:
#@title Setup and imports {display-mode: "form"}
import os, io, re, json, glob, zipfile, itertools, string, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist, squareform
from scipy.cluster.hierarchy import linkage, dendrogram
from Bio.PDB import PDBParser, MMCIFParser, NeighborSearch, PDBIO
from Bio.PDB.SASA import ShrakeRupley
from Bio.PDB.Model import Model
from Bio.PDB.Chain import Chain

# colours used for the different tools in all the plots
TOOL_COLORS = {'colabfold': '#1f77b4', 'af3': '#d62728', 'boltz': '#2ca02c', 'protenix': '#9467bd',
               'other': '#7f7f7f'}
# tools whose B-factor column contains the pLDDT
PLDDT_IN_BFACTOR = {'colabfold', 'af3', 'boltz', 'protenix'}

In [ ]:
#@title Sequences from UniProt and the AlphaFold Database {display-mode: "form"}
UNIPROT_REST = 'https://rest.uniprot.org/uniprotkb/{}.json'

def parse_uniprot_entry(d):
    """Extract the information we need from a UniProt JSON entry."""
    seq = d['sequence']['value']
    desc = d.get('proteinDescription', {})
    rec = desc.get('recommendedName') or (desc.get('submissionNames') or [{}])[0]
    genes = d.get('genes') or [{}]
    feats = []
    for f in d.get('features', []):
        loc = f.get('location', {})
        try:
            s, e = int(loc['start']['value']), int(loc['end']['value'])
        except (KeyError, TypeError, ValueError):
            continue
        feats.append({'type': f.get('type', ''), 'start': s, 'end': e, 'description': f.get('description', '')})
    return {'acc': d.get('primaryAccession', ''), 'id': d.get('uniProtkbId', ''),
            'name': rec.get('fullName', {}).get('value', ''),
            'gene': genes[0].get('geneName', {}).get('value', ''),
            'organism': d.get('organism', {}).get('scientificName', ''),
            'length': len(seq), 'sequence': seq, 'features': feats}

def fetch_uniprot(acc):
    """Download a UniProt entry (sequence and annotations)."""
    import requests
    r = requests.get(UNIPROT_REST.format(acc), timeout=30)
    if r.status_code != 200:
        raise RuntimeError(f'Could not download the UniProt entry {acc} (HTTP {r.status_code}): '
                           'check the accession number and the internet connection.')
    return parse_uniprot_entry(r.json())

def fetch_afdb_plddt(acc):
    """pLDDT per residue of the AlphaFold Database model of a UniProt entry (None if not available)."""
    import requests
    try:
        r = requests.get(f'https://alphafold.ebi.ac.uk/api/prediction/{acc}', timeout=30)
        r.raise_for_status()
        txt = requests.get(r.json()[0]['pdbUrl'], timeout=60).text
    except Exception as e:
        print(f'No AlphaFold Database model available for {acc} ({e})')
        return None
    return np.array([float(l[60:66]) for l in txt.splitlines() if l.startswith('ATOM') and l[12:16] == ' CA '])

FEATURE_COLORS = {'Domain': '#4c72b0', 'Region': '#dd8452', 'Zinc finger': '#55a868', 'Motif': '#c44e52',
                  'Coiled coil': '#8172b3', 'Transmembrane': '#937860', 'Repeat': '#da8bc3',
                  'DNA binding': '#8c8c8c', 'Compositional bias': '#ccb974'}

def plot_component(info, plddt=None, start=1, end=None, ax=None):
    """Map of one protein: UniProt features, AlphaFold DB pLDDT and the construct that you will model."""
    end = end or info['length']
    if ax is None:
        fig, ax = plt.subplots(figsize=(9, 3))
    ax.set_title(f"{info['acc']}  {info['name']}  ({info['organism']}, {info['length']} aa)", fontsize=10, loc='left')
    rows = {}
    for f in info['features']:
        if f['type'] in FEATURE_COLORS:
            row = rows.setdefault(f['type'], len(rows))
            ax.barh(-10 - 8*row, f['end']-f['start']+1, left=f['start']-1, height=6,
                    color=FEATURE_COLORS[f['type']], edgecolor='k', linewidth=0.5)
            if f['end']-f['start'] > 0.08*info['length'] and f['description']:
                ax.text((f['start']+f['end'])/2, -10-8*row, f['description'][:20], ha='center', va='center',
                        fontsize=7, color='w')
    if plddt is not None:
        ax.plot(np.arange(1, len(plddt)+1), plddt, 'k', lw=1, label='AlphaFold DB pLDDT')
        ax.axhline(70, color='grey', ls=':', lw=0.8)
    ax.axvspan(start, end, color='gold', alpha=0.25, label='construct', zorder=0)
    ax.legend(loc='lower right', bbox_to_anchor=(1.0, 1.0), fontsize=8, frameon=False, ncol=2)
    ax.set_yticks([100, 50, 0] + [-10-8*i for i in range(len(rows))])
    ax.set_yticklabels(['100', '50', '0'] + list(rows.keys()), fontsize=8)
    ax.set_xlim(0, info['length'])
    ax.set_xlabel('residue number')
    return ax

In [ ]:
#@title Define the system and prepare the inputs of the different tools {display-mode: "form"}
def build_system(system, soft_limit=1000):
    """Download the sequences, cut the constructs and give a chain ID to every copy."""
    letters = iter(string.ascii_uppercase + string.ascii_lowercase)
    chains, infos, rows = [], {}, []
    for comp in system['components']:
        acc = str(comp.get('uniprot', ''))
        if not acc or 'FILL' in acc.upper():
            raise ValueError(f"Set the UniProt accession of component '{comp.get('name')}' in the system definition.")
        info = fetch_uniprot(acc)
        infos[comp['name']] = info
        s, e = comp.get('start') or 1, comp.get('end') or info['length']
        seq = info['sequence'][s-1:e]
        for _ in range(comp.get('copies', 1)):
            chains.append({'id': next(letters), 'component': comp['name'], 'uniprot': acc,
                           'start': s, 'end': e, 'sequence': seq})
        rows.append({'component': comp['name'], 'UniProt': acc, 'protein': info['name'], 'organism': info['organism'],
                     'construct': f'{s}-{e}', 'length': len(seq), 'copies': comp.get('copies', 1)})
    total = sum(len(c['sequence']) for c in chains)
    if total > soft_limit:
        print(f'WARNING: {total} residues in total. Free Colab GPUs and the AF3 server may struggle above ~{soft_limit}: '
              'consider shorter constructs.')
    return chains, infos, pd.DataFrame(rows)

def write_inputs(chains, ions, name, outdir='inputs'):
    """Write the input of ColabFold, AlphaFold Server and Boltz for the same system."""
    os.makedirs(outdir, exist_ok=True)
    paths = {}
    # ColabFold: the sequences separated by ':'
    paths['colabfold'] = f'{outdir}/{name}_colabfold.txt'
    open(paths['colabfold'], 'w').write(':'.join(c['sequence'] for c in chains) + '\n')
    # a FASTA file with one record per chain
    paths['fasta'] = f'{outdir}/{name}.fasta'
    open(paths['fasta'], 'w').write(''.join(f">{name}_{c['id']}_{c['component']}\n{c['sequence']}\n" for c in chains))
    # Boltz: yaml file
    lines = ['version: 1', 'sequences:']
    used = [c['id'] for c in chains]
    spare = iter([x for x in string.ascii_uppercase + string.ascii_lowercase if x not in used])
    for c in chains:
        lines += ['  - protein:', f"      id: {c['id']}", f"      sequence: {c['sequence']}"]
    for ion in ions:
        for _ in range(ion.get('copies', 1)):
            lines += ['  - ligand:', f'      id: {next(spare)}', f"      ccd: {ion['ccd']}"]
    paths['boltz'] = f'{outdir}/{name}_boltz.yaml'
    open(paths['boltz'], 'w').write('\n'.join(lines) + '\n')
    # AlphaFold Server: json file (optional, you can also paste the sequences by hand)
    entities, seen = [], {}
    for c in chains:
        if c['component'] in seen:
            seen[c['component']]['proteinChain']['count'] += 1
        else:
            seen[c['component']] = {'proteinChain': {'sequence': c['sequence'], 'count': 1}}
            entities.append(seen[c['component']])
    for ion in ions:
        entities.append({'ion': {'ion': ion['ccd'], 'count': ion.get('copies', 1)}})
    paths['af3'] = f'{outdir}/{name}_af3server.json'
    json.dump([{'name': name, 'modelSeeds': [], 'sequences': entities, 'dialect': 'alphafoldserver', 'version': 1}],
              open(paths['af3'], 'w'), indent=2)
    print('ColabFold query_sequence (copy it in the notebook):')
    print(open(paths['colabfold']).read().strip())
    print('\nAlphaFold Server / Protenix: add one entity per component (copies in brackets):')
    for comp, e in seen.items():
        print(f"  {comp} [{e['proteinChain']['count']}x] ({len(e['proteinChain']['sequence'])} aa): {e['proteinChain']['sequence']}")
    for ion in ions:
        print(f"  ion {ion['ccd']} [{ion.get('copies', 1)}x]")
    print('\nFiles written:', ', '.join(paths.values()))
    return paths

def shuffled(seq, seed=0):
    """A sequence with the same composition but random order: a negative control."""
    l = list(seq)
    np.random.default_rng(seed).shuffle(l)
    return ''.join(l)

def apply_mutations(seq, mutations):
    """mutations: {position in the construct (1 = first residue): new amino acid}."""
    l = list(seq)
    for pos, aa in mutations.items():
        l[pos-1] = aa
    return ''.join(l)

def make_control(chains, component, shuffle=False, mutations=None, replacement=None, seed=0):
    """Copy of the chains in which one component is replaced by another sequence, shuffled or mutated."""
    new = []
    for c in chains:
        c = dict(c)
        if c['component'] == component:
            if replacement:
                c['sequence'] = replacement
            if shuffle:
                c['sequence'] = shuffled(c['sequence'], seed)
            if mutations:
                c['sequence'] = apply_mutations(c['sequence'], mutations)
        new.append(c)
    return new

In [ ]:
#@title Secondary structure prediction (S4PRED output) {display-mode: "form"}
def read_ss2(path):
    """Read an S4PRED (PSIPRED-like 'ss2') file."""
    return pd.read_csv(path, sep=r'\s+', skiprows=2, names=['position', 'sequence', 'ss', 'coil', 'helix', 'extended'])

def plot_ss2(df, title='', ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 2.5))
    ax.plot(df['position'], df['helix'], color='tab:blue', label='helix')
    ax.plot(df['position'], df['extended'], color='tab:red', label='strand')
    ax.plot(df['position'], df['coil'], color='k', lw=0.6, alpha=0.5, label='coil')
    ax.set_xlabel('residue'); ax.set_ylabel('probability'); ax.set_title(title, fontsize=10); ax.legend(fontsize=7)
    return ax

In [ ]:
#@title Read the predictions of the different tools {display-mode: "form"}
def tool_kind(folder):
    f = folder.lower()
    if 'af3' in f or 'alphafold3' in f or 'alphafoldserver' in f or 'alphafold_server' in f:
        return 'af3'
    if 'colabfold' in f or 'af2' in f or 'alphafold2' in f:
        return 'colabfold'
    if 'boltz' in f:
        return 'boltz'
    if 'protenix' in f:
        return 'protenix'
    return 'other'

def unpack_zip(zip_path, outdir):
    """Unzip the results of a tool in a folder (results/<tool>/)."""
    os.makedirs(outdir, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(outdir)
    print(f'{zip_path} -> {outdir}')

def unpack_results(root='results'):
    """Unzip all the .zip files found in results/<tool>/ (drag the zip files downloaded from the servers there)."""
    n = 0
    for tool in sorted(os.listdir(root)):
        for z in glob.glob(os.path.join(root, tool, '*.zip')):
            unpack_zip(z, os.path.join(root, tool)); n += 1
    print(f'{n} zip file(s) unpacked' if n else 'no zip files found in results/<tool>/')

def find_structure_files(folder, kind):
    files = sorted(glob.glob(os.path.join(folder, '**', '*.pdb'), recursive=True) +
                   glob.glob(os.path.join(folder, '**', '*.cif'), recursive=True))
    if kind == 'colabfold':
        un = [f for f in files if '_unrelaxed_rank_' in f]
        files = un or [f for f in files if '_relaxed_rank_' in f]
    elif kind in ('af3', 'boltz', 'protenix'):
        files = [f for f in files if re.search(r'_model_\d+\.(cif|pdb)$', f)] or files
    return files

def _read_json(p):
    try:
        return json.load(open(p))
    except Exception:
        return None

def read_confidence(path, kind):
    """Confidence scores written by the tool next to the structure (everything optional)."""
    conf = {'ptm': np.nan, 'iptm': np.nan, 'ranking': np.nan, 'pae': None, 'plddt': None, 'token_chains': None}
    d, base = os.path.dirname(path), os.path.basename(path)
    if kind == 'colabfold':
        js = _read_json(os.path.join(d, os.path.splitext(re.sub(r'_(un)?relaxed_', '_scores_', base))[0] + '.json'))
        if js:
            conf['ptm'], conf['iptm'] = js.get('ptm', np.nan), js.get('iptm', np.nan)
            if 'pae' in js: conf['pae'] = np.array(js['pae'], dtype=float)
            if 'plddt' in js: conf['plddt'] = np.array(js['plddt'], dtype=float)
    elif kind == 'af3':
        m = re.match(r'^(.*)_model_(\d+)\.(cif|pdb)$', base)
        if m:
            summ = _read_json(os.path.join(d, f'{m.group(1)}_summary_confidences_{m.group(2)}.json'))
            full = _read_json(os.path.join(d, f'{m.group(1)}_full_data_{m.group(2)}.json'))
            if summ:
                conf['ptm'], conf['iptm'] = summ.get('ptm', np.nan), summ.get('iptm', np.nan)
                conf['ranking'] = summ.get('ranking_score', np.nan)
            if full:
                if 'pae' in full: conf['pae'] = np.array(full['pae'], dtype=float)
                conf['token_chains'] = full.get('token_chain_ids')
    elif kind in ('boltz', 'protenix'):
        stem = os.path.splitext(base)[0]
        js = _read_json(os.path.join(d, f'confidence_{stem}.json'))
        if js:
            conf['ptm'], conf['iptm'] = js.get('ptm', np.nan), js.get('iptm', np.nan)
            conf['ranking'] = js.get('confidence_score', np.nan)
        for key in ('pae', 'plddt'):
            p = os.path.join(d, f'{key}_{stem}.npz')
            if os.path.exists(p):
                z = np.load(p)
                conf[key] = np.array(z[key] if key in z.files else z[z.files[0]], dtype=float)
    return conf

def _rep_atom(res, names=('CA', "C1'", 'P')):
    for n in names:
        if n in res: return res[n].coord
    return np.mean([a.coord for a in res], axis=0)

def _cb_atom(res):
    for n in ('CB', 'CA', "C1'", 'P'):
        if n in res: return res[n].coord
    return np.mean([a.coord for a in res], axis=0)

def load_model(path, tool, kind, label=None):
    """Read a structure and its confidence scores into a dictionary."""
    parser = MMCIFParser(QUIET=True) if path.lower().endswith(('.cif', '.mmcif')) else PDBParser(QUIET=True)
    model = parser.get_structure('m', path)[0]
    chains = []
    for ch in model:
        res = [r for r in ch if r.id[0] == ' ']
        if not res: continue
        chains.append({'id': ch.id, 'chain': ch, 'residues': res, 'resnames': [r.get_resname().strip() for r in res],
                       'resnums': [r.id[1] for r in res], 'ca': np.array([_rep_atom(r) for r in res]),
                       'cb': np.array([_cb_atom(r) for r in res])})
    conf = read_confidence(path, kind)
    n_res = sum(len(c['residues']) for c in chains)
    # pLDDT per residue: from the B-factor column (AlphaFold-like tools) or from the tool's own file
    pl = np.full(n_res, np.nan)
    if kind in PLDDT_IN_BFACTOR:
        pl = np.array([np.mean([a.bfactor for a in r]) for c in chains for r in c['residues']])
        if conf['plddt'] is not None and len(conf['plddt']) == n_res:
            pl = conf['plddt']
        if np.nanmax(pl) <= 1.0:
            pl = pl*100
    # PAE: keep only the polymer residues
    pae = conf['pae']
    if pae is not None and pae.shape[0] != n_res:
        if conf['token_chains'] is not None and len(conf['token_chains']) == pae.shape[0]:
            keep = np.isin(np.array(conf['token_chains']), [c['id'] for c in chains])
            pae = pae[np.ix_(keep, keep)]
        elif pae.shape[0] > n_res:
            pae = pae[:n_res, :n_res]
    if pae is not None and pae.shape[0] != n_res:
        print(f'  PAE of {os.path.basename(path)} has a different size ({pae.shape[0]} vs {n_res} residues): ignored')
        pae = None
    k = 0
    for c in chains:
        c['plddt'] = pl[k:k+len(c['residues'])]
        k += len(c['residues'])
    return {'label': label or os.path.basename(path), 'tool': tool, 'kind': kind, 'path': path, 'chains': chains,
            'conf': conf, 'pae': pae, 'n_res': n_res, 'interface': None}

def load_models(root='results', verbose=True):
    """Load all the models found in results/<tool>/ (one sub-folder per tool)."""
    if not os.path.isdir(root):
        raise FileNotFoundError(f"The folder '{root}' does not exist: put the results of every tool in {root}/<tool name>/")
    models = []
    for tool in sorted(os.listdir(root)):
        folder = os.path.join(root, tool)
        if not os.path.isdir(folder): continue
        kind = tool_kind(tool)
        for i, f in enumerate(find_structure_files(folder, kind)):
            m = re.search(r'rank_(\d+)', f) if kind == 'colabfold' else re.search(r'_model_(\d+)', f)
            idx = int(m.group(1)) if m else i + 1
            try:
                models.append(load_model(f, tool, kind, label=f'{tool}_{idx}'))
            except Exception as e:
                print(f'  skipped {f}: {e}')
    if verbose:
        print(pd.Series([m['tool'] for m in models]).value_counts().to_string() if models else 'no models found')
    if not models:
        print(f'No models found: check that the structure files (.pdb or .cif) are in {root}/<tool name>/')
    return models

def check_compatible(models):
    """All the models must contain the same chains with the same residues."""
    ref = sorted(tuple(c['resnames']) for c in models[0]['chains'])
    for m in models[1:]:
        if sorted(tuple(c['resnames']) for c in m['chains']) != ref:
            raise ValueError(f"{m['label']} has chains of length {[len(c['residues']) for c in m['chains']]} but "
                             f"{models[0]['label']} has {[len(c['residues']) for c in models[0]['chains']]}: "
                             'the predictions must be made with the same constructs.')
    return True

def compatible_models(models):
    """Keep the models that have the most common set of chains (same constructs); report the others."""
    if not models:
        print('Nothing to compare yet: run the predictions and put their results in results/<tool name>/, '
              'then run this cell again.')
        return [], []
    sig = lambda m: tuple(sorted(tuple(c['resnames']) for c in m['chains']))
    counts = pd.Series([hash(sig(m)) for m in models]).value_counts()
    keep_hash = counts.index[0]
    kept = [m for m in models if hash(sig(m)) == keep_hash]
    dropped = [m for m in models if hash(sig(m)) != keep_hash]
    for m in dropped:
        print(f"set aside {m['label']}: chains of length {[len(c['residues']) for c in m['chains']]}, "
              f"the other models have {[len(c['residues']) for c in kept[0]['chains']]}")
    return kept, dropped

In [ ]:
#@title Interface analysis of one model {display-mode: "form"}
def pdockq(mean_plddt, n_contacts):
    """Approximate pDockQ score (Bryant et al. 2022): from interface pLDDT and number of interface contacts."""
    if not n_contacts or np.isnan(mean_plddt):
        return np.nan
    x = mean_plddt*np.log10(n_contacts)
    return 0.724/(1 + np.exp(-0.052*(x - 152.611))) + 0.018

_sr = ShrakeRupley()
def _sasa(chain_dicts):
    m = Model(0)
    for c in chain_dicts:
        new = Chain(c['id'])
        for r in c['residues']:
            new.add(r.copy())
        m.add(new)
    _sr.compute(m, level='A')
    return sum(a.sasa for a in m.get_atoms())

def interface_analysis(md, contact_cutoff=5.0, cb_cutoff=8.0, with_area=True):
    """Contacts and interface properties between every pair of chains of a model."""
    if md['interface'] is not None:
        return md['interface']
    chains = md['chains']
    where, atoms = {}, []
    for ci, c in enumerate(chains):
        for ri, r in enumerate(c['residues']):
            where[id(r)] = (ci, ri)
            atoms += [a for a in r if a.element != 'H']
    contacts = set()
    for r1, r2 in NeighborSearch(atoms).search_all(contact_cutoff, level='R'):
        (c1, i1), (c2, i2) = where[id(r1)], where[id(r2)]
        if c1 != c2:
            contacts.add(((c1, i1), (c2, i2)) if c1 < c2 else ((c2, i2), (c1, i1)))
    offsets = np.cumsum([0] + [len(c['residues']) for c in chains])
    rows, alone = [], {}
    for ci, cj in itertools.combinations(range(len(chains)), 2):
        pair = [(a, b) for (a, b) in contacts if a[0] == ci and b[0] == cj]
        if not pair: continue
        d = cdist(chains[ci]['cb'], chains[cj]['cb'])
        ii, jj = np.where(d < cb_cutoff)
        pl = np.concatenate([chains[ci]['plddt'][np.unique(ii)], chains[cj]['plddt'][np.unique(jj)]])
        mean_pl = float(np.mean(pl)) if len(pl) else np.nan
        row = {'chain_i': chains[ci]['id'], 'chain_j': chains[cj]['id'], 'contacts': len(pair),
               'res_i': len({a[1] for a, b in pair}), 'res_j': len({b[1] for a, b in pair}),
               'cb_contacts': len(ii), 'interface_plddt': mean_pl, 'pDockQ': pdockq(mean_pl, len(ii))}
        if md['pae'] is not None:
            oi, oj = offsets[ci], offsets[cj]
            blk = np.concatenate([md['pae'][oi:oi+len(chains[ci]['residues']), oj:oj+len(chains[cj]['residues'])].ravel(),
                                  md['pae'][oj:oj+len(chains[cj]['residues']), oi:oi+len(chains[ci]['residues'])].ravel()])
            row['interface_pae'] = float(blk.mean())
        else:
            row['interface_pae'] = np.nan
        row['area'] = np.nan
        if with_area:
            try:
                for k in (ci, cj):
                    if k not in alone: alone[k] = _sasa([chains[k]])
                row['area'] = (alone[ci] + alone[cj] - _sasa([chains[ci], chains[cj]]))/2
            except Exception:
                pass
        rows.append(row)
    md['interface'] = {'contacts': contacts, 'pairs': pd.DataFrame(rows)}
    return md['interface']

def summarize(models, with_area=True):
    """One row per model: confidence scores and interface properties."""
    if not models:
        print('No models loaded: nothing to summarise.')
        return pd.DataFrame(), pd.DataFrame()
    rows, pairs = [], []
    for md in models:
        it = interface_analysis(md, with_area=with_area)
        p = it['pairs'].copy()
        p.insert(0, 'model', md['label']); p.insert(1, 'tool', md['tool'])
        pairs.append(p)
        best = p.sort_values('contacts', ascending=False).iloc[0] if len(p) else None
        rows.append({'model': md['label'], 'tool': md['tool'], 'pTM': md['conf']['ptm'], 'ipTM': md['conf']['iptm'],
                     'mean_pLDDT': (lambda v: float(np.mean(v[~np.isnan(v)])) if (~np.isnan(v)).any() else np.nan)(
                         np.concatenate([c['plddt'] for c in md['chains']])),
                     'interface_contacts': len(it['contacts']),
                     'interface_area': float(p['area'].sum()) if len(p) else 0.0,
                     'interface_pLDDT': float(best['interface_plddt']) if best is not None else np.nan,
                     'interface_PAE': float(best['interface_pae']) if best is not None else np.nan,
                     'pDockQ': float(best['pDockQ']) if best is not None else np.nan})
    return pd.DataFrame(rows), (pd.concat(pairs, ignore_index=True) if pairs else pd.DataFrame())

In [ ]:
#@title Comparison of models {display-mode: "form"}
def _sig(c): return tuple(c['resnames'])

def chain_permutations(ref, md, limit=720):
    """Ways of matching the chains of md with those of ref (identical chains can be exchanged)."""
    groups = {}
    for i, c in enumerate(ref['chains']): groups.setdefault(_sig(c), []).append(i)
    mgroups = {}
    for i, c in enumerate(md['chains']): mgroups.setdefault(_sig(c), []).append(i)
    keys = list(groups)
    options = [list(itertools.permutations(mgroups[k])) for k in keys]
    perms = []
    for combo in itertools.product(*options):
        perm = [None]*len(ref['chains'])
        for k, mapped in zip(keys, combo):
            for ref_i, m_i in zip(groups[k], mapped):
                perm[ref_i] = m_i
        perms.append(perm)
        if len(perms) >= limit: break
    return perms

def _remap_contacts(contacts, perm):
    inv = {m_i: ref_i for ref_i, m_i in enumerate(perm)}
    out = set()
    for (c1, i1), (c2, i2) in contacts:
        a, b = (inv[c1], i1), (inv[c2], i2)
        out.add((a, b) if a[0] < b[0] else (b, a))
    return out

def contact_similarity(ref, md):
    """Jaccard index of the sets of inter-chain residue contacts (1 = same interface). Returns (value, best chain matching)."""
    cr = interface_analysis(ref, with_area=False)['contacts']
    cm = interface_analysis(md, with_area=False)['contacts']
    best, best_perm = -1, None
    for perm in chain_permutations(ref, md):
        s = _remap_contacts(cm, perm)
        j = len(cr & s)/len(cr | s) if (cr | s) else 1.0
        if j > best: best, best_perm = j, perm
    return best, best_perm

def kabsch(P, Q):
    """Rotation R and translation t that best superimpose P on Q (Q ~ P R^T + t)."""
    Pc, Qc = P.mean(0), Q.mean(0)
    U, S, Vt = np.linalg.svd((P - Pc).T @ (Q - Qc))
    D = np.diag([1, 1, np.sign(np.linalg.det(Vt.T @ U.T))])
    R = Vt.T @ D @ U.T
    return R, Qc - R @ Pc

def lrmsd(ref, md, fixed=None, plddt_min=None):
    """Superimpose md on ref using the chain 'fixed' (default: the longest) and return the CA RMSD of all the other chains
    (the best over the possible matchings of identical chains). Returns (rmsd, perm)."""
    n = len(ref['chains'])
    fixed = int(np.argmax([len(c['residues']) for c in ref['chains']])) if fixed is None else fixed
    best, best_perm = np.inf, None
    for perm in chain_permutations(ref, md):
        masks = []
        for i in range(n):
            mk = np.ones(len(ref['chains'][i]['residues']), bool)
            if plddt_min is not None:     # models without pLDDT (NaN) are not filtered
                for v in (ref['chains'][i]['plddt'], md['chains'][perm[i]]['plddt']):
                    mk &= np.isnan(v) | (v >= plddt_min)
            masks.append(mk)
        if not masks[fixed].any(): continue
        P = md['chains'][perm[fixed]]['ca'][masks[fixed]]
        Q = ref['chains'][fixed]['ca'][masks[fixed]]
        R, t = kabsch(P, Q)
        d2 = [np.sum((md['chains'][perm[i]]['ca'][masks[i]] @ R.T + t - ref['chains'][i]['ca'][masks[i]])**2, axis=1)
              for i in range(n) if i != fixed and masks[i].any()]
        if not d2: continue
        val = float(np.sqrt(np.concatenate(d2).mean()))
        if val < best: best, best_perm = val, perm
    return best, best_perm

def compare_models(models, plddt_min=None):
    """Matrices of contact similarity (Jaccard, high = similar) and L-RMSD (Angstrom, low = similar)."""
    if len(models) < 2:
        print('At least two models are needed to compare them.')
        return pd.DataFrame(), pd.DataFrame()
    check_compatible(models)
    labels = [m['label'] for m in models]
    n = len(models)
    J, L = np.eye(n), np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            J[i, j] = J[j, i] = contact_similarity(models[i], models[j])[0]
            L[i, j] = L[j, i] = lrmsd(models[i], models[j], plddt_min=plddt_min)[0]
    return pd.DataFrame(J, labels, labels), pd.DataFrame(L, labels, labels)

def agreement_table(J, L, models):
    """Average agreement between the models of every pair of tools (and between the models of the same tool)."""
    tools = list(dict.fromkeys(m['tool'] for m in models))
    tool_of = {m['label']: m['tool'] for m in models}
    rows = []
    for a, b in itertools.combinations_with_replacement(tools, 2):
        la = [l for l in J.index if tool_of[l] == a]; lb = [l for l in J.index if tool_of[l] == b]
        pairs = [(x, y) for x in la for y in lb if x != y and (a != b or x < y)]
        if not pairs: continue
        rows.append({'tool 1': a, 'tool 2': b, 'n pairs': len(pairs),
                     'shared contacts (Jaccard)': np.mean([J.loc[x, y] for x, y in pairs]),
                     'L-RMSD (A)': np.mean([L.loc[x, y] for x, y in pairs])})
    return pd.DataFrame(rows)

def plot_similarity(J, L, models, figsize=(13, 5.5)):
    """Heat maps of the comparison, models ordered by clustering of the interface contacts."""
    n = len(J)
    order = dendrogram(linkage(squareform(1 - J.values, checks=False), 'average'), no_plot=True)['leaves'] if n > 2 else list(range(n))
    labs = [J.index[i] for i in order]
    kinds = {m['label']: m['kind'] for m in models}
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    for ax, M, title, cmap, kw in [(axes[0], J, 'interface contacts shared (Jaccard)', 'viridis', {'vmin': 0, 'vmax': 1}),
                                   (axes[1], L, 'L-RMSD (A), superposed on the longest chain', 'viridis_r',
                                    {'vmin': 0, 'vmax': max(10, np.percentile(L.values[np.isfinite(L.values)], 95))})]:
        im = ax.imshow(M.values[np.ix_(order, order)], cmap=cmap, **kw)
        ax.set_xticks(range(n)); ax.set_yticks(range(n))
        ax.set_xticklabels(labs, rotation=90, fontsize=7); ax.set_yticklabels(labs, fontsize=7)
        for tl in list(ax.get_xticklabels()) + list(ax.get_yticklabels()):
            tl.set_color(TOOL_COLORS.get(kinds[tl.get_text()], 'k'))
        ax.set_title(title, fontsize=10)
        plt.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    return order

def consensus_interface(models, ref=0, system_chains=None):
    """For every residue of the reference model: in which fraction of the models is it at the interface.
    system_chains (output of build_system) adds the residue number of the UniProt entry."""
    ref_md = models[ref]
    count = {}
    for md in models:
        perm = contact_similarity(ref_md, md)[1]
        for (a, b) in _remap_contacts(interface_analysis(md, with_area=False)['contacts'], perm):
            count.setdefault(a, set()).add(md['label']); count.setdefault(b, set()).add(md['label'])
    rows = []
    for ci, c in enumerate(ref_md['chains']):
        for ri in range(len(c['residues'])):
            frac = len(count.get((ci, ri), ()))/len(models)
            row = {'chain': c['id'], 'residue': ri + 1, 'name': c['resnames'][ri], 'fraction_of_models': frac,
                   'pLDDT_ref': c['plddt'][ri]}
            if system_chains is not None and ci < len(system_chains):
                row['uniprot_pos'] = system_chains[ci]['start'] + ri
                row['component'] = system_chains[ci]['component']
            rows.append(row)
    return pd.DataFrame(rows)

def plot_consensus(cons, threshold=0.8):
    """Fraction of the models in which each residue is at the interface, chain by chain."""
    groups = list(cons.groupby('chain', sort=False))
    fig, axes = plt.subplots(len(groups), 1, figsize=(10, 1.9*len(groups)), squeeze=False)
    for ax, (cid, g) in zip(axes[:, 0], groups):
        ax.bar(g['residue'], g['fraction_of_models'], width=1.0, color='tab:blue')
        ax.axhline(threshold, color='k', ls='--', lw=0.8)
        ax.set_ylim(0, 1); ax.set_ylabel(f'chain {cid}', fontsize=9)
    axes[-1, 0].set_xlabel('residue number in the construct')
    axes[0, 0].set_title('fraction of the models in which the residue is at the interface', fontsize=10)
    plt.tight_layout()
    return axes

In [ ]:
#@title Plots and 3D views {display-mode: "form"}
def _boundaries(md):
    return np.cumsum([len(c['residues']) for c in md['chains']])[:-1]

def plot_plddt(models, ax=None):
    if not models:
        print('No models loaded.'); return None
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 3))
    models = [m for m in models if not np.isnan(np.concatenate([c['plddt'] for c in m['chains']])).all()]
    for md in models:
        v = np.concatenate([c['plddt'] for c in md['chains']])
        ax.plot(v, color=TOOL_COLORS.get(md['kind'], 'k'), alpha=0.6, lw=1)
    for b in _boundaries(models[0]): ax.axvline(b, color='k', ls='--')
    for k in sorted({m['kind'] for m in models}): ax.plot([], [], color=TOOL_COLORS.get(k, 'k'), label=k)
    ax.axhline(70, color='grey', ls=':'); ax.set_ylim(0, 100)
    ax.set_xlabel('residue (chains one after the other)'); ax.set_ylabel('pLDDT'); ax.legend(fontsize=8)
    return ax

def plot_pae(md, ax=None, vmax=30):
    if md['pae'] is None:
        print(f"{md['label']}: no PAE available"); return None
    if ax is None:
        fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(md['pae'], cmap='Greens_r', vmin=0, vmax=vmax)
    for b in _boundaries(md):
        ax.axvline(b-0.5, color='k', lw=0.8); ax.axhline(b-0.5, color='k', lw=0.8)
    ax.set_title(f"{md['label']}   ipTM={md['conf']['iptm']:.2f}" if not np.isnan(md['conf']['iptm']) else md['label'], fontsize=9)
    plt.colorbar(im, ax=ax, shrink=0.8, label='expected error (A)')
    return ax

def plot_scores(tables, columns=('ipTM', 'pDockQ', 'interface_area', 'interface_PAE')):
    """Compare the scores of the models. tables: DataFrame or {name: DataFrame}."""
    if isinstance(tables, pd.DataFrame): tables = {'': tables}
    fig, axes = plt.subplots(1, len(columns), figsize=(3.6*len(columns), 3.4))
    x0 = 0
    ticks, labels = [], []
    for name, df in tables.items():
        for tool, g in df.groupby('tool', sort=False):
            for col, ax in zip(columns, axes):
                ax.plot(x0 + np.arange(len(g)), g[col].values, 'o', color=TOOL_COLORS.get(tool_kind(tool), 'k'))
            ticks.append(x0 + (len(g)-1)/2); labels.append(f'{name}\n{tool}' if name else tool)
            x0 += len(g) + 1
    for col, ax in zip(columns, axes):
        ax.set_title(col, fontsize=10); ax.set_xticks(ticks); ax.set_xticklabels(labels, rotation=60, fontsize=7)
        if col in ('pTM', 'ipTM', 'pDockQ'): ax.set_ylim(0, 1)      # scores between 0 and 1: do not zoom on the noise
        else: ax.set_ylim(bottom=0)
    plt.tight_layout()
    return axes

def _transformed_pdb(md, R, t):
    new = [(a, a.coord.copy()) for a in md['chains'][0]['chain'].get_parent().get_atoms()]
    for a, c in new:
        a.coord = (R @ c + t).astype('float32')
    s = io.StringIO()
    io_ = PDBIO(); io_.set_structure(md['chains'][0]['chain'].get_parent().get_parent()); io_.save(s)
    for a, c in new: a.coord = c
    return s.getvalue()

def show_models(models, ref=0, color='tool', fixed=None, width=700, height=500):
    """3D view of one or more models superposed on one chain of the reference model.
    color: 'tool', 'chain' or 'plddt'."""
    import py3Dmol
    view = py3Dmol.view(width=width, height=height)
    r = models[ref]
    fixed = int(np.argmax([len(c['residues']) for c in r['chains']])) if fixed is None else fixed
    chain_colors = ['#4c72b0', '#dd8452', '#55a868', '#c44e52', '#8172b3', '#937860']
    for k, md in enumerate(models):
        perm = lrmsd(r, md, fixed=fixed)[1] if md is not r else list(range(len(r['chains'])))
        R, t = kabsch(md['chains'][perm[fixed]]['ca'], r['chains'][fixed]['ca'])
        view.addModel(_transformed_pdb(md, R, t), 'pdb')
        if color == 'plddt':
            view.setStyle({'model': k}, {'cartoon': {'colorscheme': {'prop': 'b', 'gradient': 'roygb', 'min': 40, 'max': 90}}})
        elif color == 'chain':
            for i, cid in enumerate(c['id'] for c in md['chains']):
                view.setStyle({'model': k, 'chain': cid}, {'cartoon': {'color': chain_colors[perm.index(i) % 6]}})
        else:
            view.setStyle({'model': k}, {'cartoon': {'color': TOOL_COLORS.get(md['kind'], '#7f7f7f')}})
    view.zoomTo()
    return view

## Part 1. The system

Your group works on one of the three systems defined below (set `GROUP`). Both are complexes of HPV proteins with host proteins **for which no experimental structure of the assembly is available**: you are not reproducing a known answer, you are making a hypothesis.

| | Group A | Group B | Group C |
|:--|:--|:--|:--|
| system | HPV16 **E2** transactivation domain (dimer) + **TopBP1** BRCT domains | HPV16 **E7** dimer + **Rb** pocket domain + Zn<sup>2+</sup> | HPV16 **E6** + **MAGI-1** PDZ1 domain + Zn<sup>2+</sup> |
| what is known | the interaction was found by yeast two-hybrid; phosphorylation of E2 Ser23 by CK2 promotes it; the E2 mutants S23A and R37A separate TopBP1 from BRD4 binding; TopBP1 BRCT6 binds the BRD4 ET domain | E7 binds the Rb pocket through the LxCxE motif of its disordered N-terminus; the CR3 domain is a Zn-binding homodimer that also contacts Rb and displaces E2F; full-length E7 binds Rb ~100x more tightly than the LxCxE part alone | E6 ends with a PDZ-binding motif (PBM) that binds the first PDZ domain of MAGI-1, which is then degraded; residues *outside* the canonical PDZ boundaries (its C-terminal extension) take part in the binding, and mutating them lowers the affinity; E6 binds two zinc ions |
| what is *not* known | how the E2 transactivation domain engages the BRCT domains, and how the three proteins are arranged together | how the whole E7 (disordered N-terminus **and** CR3 dimer) engages the Rb pocket at the same time | how the **whole** E6 protein, and not only its last residues, is arranged against the PDZ domain: no structure of the full PDZ1-E6 complex is available |
| partial answer keys (use at the end, not at the beginning!) | the E2 TAD **dimer**: PDB [1DTO](https://www.rcsb.org/structure/1DTO); the E2 TAD bound to the BRD4 **C-terminal motif**: PDB [2NNU](https://www.rcsb.org/structure/2NNU); TopBP1 **BRCT0-1-2** alone: PDB [3AL2/6RML](https://www.rcsb.org/structure/6RML) | Rb pocket with the E7 **LxCxE peptide**: PDB [1GUX](https://www.rcsb.org/structure/1GUX); E7 CR3 dimer: PDB [2B9D](https://www.rcsb.org/structure/2B9D) | the MAGI-1 PDZ1 domain with the **last 11 residues** of E6: PDB [2KPL](https://www.rcsb.org/structure/2KPL) (and the free domain: [2KPK](https://www.rcsb.org/structure/2KPK)); the fold of E6 itself, inside a *different* complex: PDB [4XR8](https://www.rcsb.org/structure/4XR8) |
| the interesting difficulty | the key phosphorylation (Ser23) cannot be modelled by most tools | a long disordered region carrying a short motif, plus a zinc-dependent homodimer | a short motif at the very end of a folded protein: does the prediction place the rest of E6 sensibly, or does it just dock the tail? |

Each system is made of **components** (proteins, with a UniProt accession number, the part of the sequence to model, and the number of copies) and possibly **ions**. The cell below downloads the sequences from UniProt and prepares the inputs of all the tools from the *same* definition, so that the predictions can be compared.

The constructs proposed below are a **starting point taken from the literature**. **After looking at the annotations you should decide your own constructs** (change `start` and `end` and run the cells again): long disordered regions are not predicted realistically and can degrade the prediction of the interface (see the summary of Lecture 4), but for these systems a disordered region may be exactly where the interaction happens, so do not trim it away without thinking. Keep the total length of the system below about 1000 residues, otherwise free GPUs and the servers may not cope with it.

In [ ]:
GROUP = 'A'       # <-- 'A', 'B' or 'C'

SYSTEMS = {
    # ---------------------------------------------------------------- Group A: E2 / TopBP1 / BRD4
    'A': {'name': 'E2_TopBP1',
          'components': [
              # HPV16 E2 (365 aa): transactivation domain (TAD), it carries Ser23 and Arg37.
              # The TAD dimerises (PDB 1DTO), so we model two copies: does the dimer form in your predictions?
              {'name': 'E2_TAD',   'uniprot': 'P03120', 'start': 1,   'end': 200,  'copies': 2},
              # human TopBP1 (1522 aa): the N-terminal triple BRCT module (BRCT0-1-2), as crystallised
              {'name': 'TopBP1_BRCT012', 'uniprot': 'Q92547', 'start': 1, 'end': 290, 'copies': 1},
              # OPTIONAL extensions, one at a time (watch the total length, and check the boundaries in the
              # UniProt annotations printed below before using them):
              #   another BRCT module of TopBP1, to test which one binds E2: BRCT4-5 (about 548-738) or BRCT6 (about 900-991)
              #   {'name': 'TopBP1_BRCT6', 'uniprot': 'Q92547', 'start': 900, 'end': 991, 'copies': 1},
              #   the ET domain of human BRD4 (O60885), that binds TopBP1 BRCT6: take its boundaries from the
              #   UniProt annotation of O60885 (the 'NET'/ET domain), and add TopBP1 BRCT6 as well, otherwise
              #   there is nothing for it to bind to
          ],
          'ions': []},
    # ---------------------------------------------------------------- Group B: E7 / Rb / zinc
    'B': {'name': 'E7_Rb',
          'components': [
              # HPV16 E7 (98 aa): the whole protein (disordered N-terminus with the LxCxE motif + CR3 domain),
              # the CR3 domain forms a dimer, so two copies
              {'name': 'E7',       'uniprot': 'P03129', 'start': 1,   'end': 98,   'copies': 2},
              # human Rb (928 aa): the pocket domain (A and B boxes), as crystallised
              {'name': 'Rb_pocket','uniprot': 'P06400', 'start': 372, 'end': 787,  'copies': 1},
          ],
          # each E7 monomer binds one zinc through its two CxxC motifs
          'ions': [{'ccd': 'ZN', 'copies': 2}]},
    # ---------------------------------------------------------------- Group C: E6 / MAGI-1 / zinc
    'C': {'name': 'E6_MAGI1',
          'components': [
              # HPV16 E6 (151 aa): the whole protein. Its last residues are the PDZ-binding motif (PBM)
              {'name': 'E6',       'uniprot': 'P03126', 'start': 1,   'end': 151,  'copies': 1},
              # human MAGI-1 (1491 aa): the first PDZ domain in the EXTENDED construct used for the NMR
              # structure (455-580): the extensions beyond the canonical domain take part in the binding
              {'name': 'MAGI1_PDZ1', 'uniprot': 'Q96QZ7', 'start': 455, 'end': 580, 'copies': 1},
              # OPTIONAL: the LxxLL motif of E6AP (UBE3A, Q05086) binds E6 on the opposite side. Adding it
              # gives a three-protein system where one interface is known and the other is not; take the
              # boundaries of the LxxLL motif from the UniProt annotations of Q05086 printed below
          ],
          # E6 binds two zinc ions, each through two CxxC motifs
          'ions': [{'ccd': 'ZN', 'copies': 2}]},
}
SYSTEM = SYSTEMS[GROUP]

In [ ]:
chains, infos, table = build_system(SYSTEM)
table

The chains are called A, B, C... in the order of the table (one for each copy). Now a map of every component: the UniProt annotations (domains, regions, motifs...), the confidence (pLDDT) of the model of the isolated protein in the **AlphaFold Database**, and the construct that you are modelling (yellow).

In [ ]:
fig, axes = plt.subplots(len(infos), 1, figsize=(9, 3.3*len(infos)), squeeze=False)
for ax, comp in zip(axes[:, 0], SYSTEM['components']):
    info = infos[comp['name']]
    plot_component(info, fetch_afdb_plddt(info['acc']), comp.get('start') or 1, comp.get('end'), ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# the annotations of UniProt (this includes known mutagenesis experiments and binding sites: useful evidence!)
skip = ['Chain', 'Natural variant', 'Sequence conflict', 'Helix', 'Beta strand', 'Turn']
for name, info in infos.items():
    print(f"\n{name}: {info['acc']} {info['gene']} - {info['name']}")
    f = pd.DataFrame(info['features'])
    if len(f):
        display(f[~f['type'].isin(skip)].reset_index(drop=True))

**Questions** (the answers go in the report)

*   Which parts of each protein are folded and which are likely disordered? (look at the pLDDT of the AlphaFold Database)
*   Which annotated domains, motifs or regions could be involved in the interaction? Is anything known (mutagenesis, binding sites) about the interaction?
*   What constructs will you model, and why? Are you keeping a disordered region because the interaction needs it, or cutting it because it would add noise?
*   **Group A**: where are Ser23 and Arg37 in the E2 construct? Can the tools you will use model a phosphorylated serine? If not, what does this mean for your prediction?
*   **Group B**: where are the LxCxE motif and the two CxxC zinc-binding motifs in E7? Which tools can model the zinc ion, and which cannot?
*   **Group C**: which are the last residues of E6 (the PDZ-binding motif), and where are the CxxC motifs that bind the two zinc ions? The MAGI-1 construct is deliberately longer than the canonical PDZ domain: which residues are the extensions, and do they end up at the interface in your models?

When you are happy with the constructs, **run again the two cells above (`SYSTEM` and `build_system`)**.

## Part 2. The components: secondary structure

We predict the secondary structure of each component from its sequence with [S4PRED](https://github.com/psipred/s4pred), a method that does not need a multiple sequence alignment.

S4PRED needs **PyTorch**. On Colab it is already installed. **On your own computer** it is included in the course environment (`environment.yml` in the main folder of the repository): if you created the environment before PyTorch was added, update it with `conda env update -f environment.yml --prune` and restart the kernel. This part is independent of the rest: you can also set `run_s4pred = False` and go on, you will only miss the secondary structure plot.

In [ ]:
run_s4pred = True

# one FASTA file with one sequence per component (useful also if you prefer to run S4PRED elsewhere)
unique = {}
for c in chains: unique.setdefault(c['component'], c['sequence'])
os.makedirs('inputs', exist_ok=True)
with open('inputs/components.fasta', 'w') as f:
    for name, seq in unique.items(): f.write(f'>{name}\n{seq}\n')

import importlib.util
if run_s4pred and importlib.util.find_spec('torch') is None:
    run_s4pred = False
    print("PyTorch is not installed in the Python of this notebook, so S4PRED cannot run here.")
    print("Locally: update the course environment (conda env update -f environment.yml --prune), restart the kernel")
    print("and run this cell again. The rest of the notebook does not need it: you can simply go on.")

if run_s4pred:
    if not os.path.exists('s4pred'):
        !git clone -q --depth 1 https://github.com/psipred/s4pred
    if not os.path.exists('s4pred/weights'):
        !wget -q -O s4pred/weights.tar.gz http://bioinfadmin.cs.ucl.ac.uk/downloads/s4pred/weights.tar.gz
        !tar -xzf s4pred/weights.tar.gz -C s4pred

    # S4PRED does not create its output folder: we do it, and we remove old predictions (of other constructs)
    os.makedirs('ss', exist_ok=True)
    for old in glob.glob('ss/s4_out_*.ss2'): os.remove(old)
    # sys.executable is the python of THIS notebook (important if you work in a virtual environment)
    # the prediction of the i-th sequence is saved in ss/s4_out_i.ss2
    !{sys.executable} s4pred/run_model.py --silent --save-files --save-by-idx --outdir {os.path.abspath('ss')} inputs/components.fasta

    if not os.path.exists('ss/s4_out_0.ss2'):
        run_s4pred = False
        print("\nS4PRED did not produce its output (look at the messages above: the download of the weights "
              "may have failed). Skipping this part, the rest of the notebook does not need it.")

In [ ]:
if run_s4pred:
    fig, axes = plt.subplots(len(unique), 1, figsize=(9, 2.7*len(unique)), squeeze=False)
    for i, (ax, name) in enumerate(zip(axes[:, 0], unique)):
        plot_ss2(read_ss2(f'ss/s4_out_{i}.ss2'), title=name, ax=ax)
    plt.tight_layout()
    plt.show()
else:
    print('secondary structure prediction not available (see the cell above)')

Compare with the pLDDT of the AlphaFold Database models and with the annotations: where are the helices and strands? Are there regions that are neither (coil), and are they confidently coil or probably disordered?

## Part 3. Homology modelling of the complex (SWISS-MODEL)

Homology modelling builds a model on the structure of a homologous protein (the *template*). **SWISS-MODEL can model complexes**, not only single chains: given the sequences of the partners, it looks for a template *assembly* and copies its interchain contacts.

*   https://swissmodel.expasy.org/ (help: https://swissmodel.expasy.org/docs/help)

This is the opposite approach to the tools of Part 4: here the interface can only be built **if it already exists in some known structure**. Doing it first tells you how much of your complex is already covered by experimental knowledge, and gives you something to compare the deep-learning predictions with.

### 3a. The whole complex at once

1.  Open https://swissmodel.expasy.org/interactive and paste the sequence of your **first** component (printed by the cell of Part 4, or in `inputs/<name>.fasta`).
2.  Click **"Add Hetero Target"** and paste the sequence of the **second** component. Repeat for the other components. *Identical copies do not need to be added twice*: if the chosen template is an oligomer, SWISS-MODEL builds the oligomer by itself (see the `Oligo-state` of the model).
3.  Give the project a name and click **Build Models**.

### 3b. Each component separately

Run SWISS-MODEL once more for **each component alone**. This tells you whether the individual folds have good templates, independently of the interface, and gives you a comparison for the chains of the complex models.

### 3c. What to write down

For every run, record in the report:

| what | where to find it | what it means |
|:--|:--|:--|
| **template**, its oligo-state and organism | table of the templates | is the template a *complex*? Is its partner related to your partner? |
| **sequence identity** and **coverage** | table of the templates | below ~30% identity the model becomes unreliable |
| **GMQE** and **QMEANDisCo** (0-1) | model card | quality of the *fold* of the chains |
| **QSQE** (0-1) | table of the templates | quality expected for the **interchain contacts**: this is the one that matters for a complex. It is computed *only* when an oligomer can be built: if it is missing, SWISS-MODEL did not find a template assembly for your complex |

**A negative result is a result.** If SWISS-MODEL does not find any template for the interface (no QSQE, or only the single chains are modelled), write it down: it means that *no experimentally known structure contains this interaction*, which is precisely why the practical uses the tools of Part 4. If instead it does build a complex, look carefully at where the interface comes from: a template assembly of homologs is real evidence, a low QSQE from a distant template is not.

Download the models you want to keep (`model.pdb` from the project page, or the zip of the project) and put them in the folder `results/swissmodel` (File explorer on the left). Models of single components and models of the complex can both go there: the analysis of Part 5 keeps the models that contain the same chains as the majority and sets the others aside, telling you which. Note that the B-factor column of SWISS-MODEL models is **not** a pLDDT (it is a QMEANDisCo estimate), so the analysis only uses their coordinates, and these models are shown in grey in the plots.

## Part 4. Predicting the complex with several tools

All the tools receive **the same constructs**. The next cell writes the input of each tool from your system definition.

In [ ]:
paths = write_inputs(chains, SYSTEM.get('ions', []), SYSTEM['name'])

**Rules of the game**

*   *Same constructs, same copies* for all the tools, otherwise the models cannot be compared.
*   **No templates** in the AlphaFold-like predictions (`template_mode = none` in ColabFold): we want to see what the network predicts from sequence and evolution. (A prediction with templates is homology modelling in disguise: you can do it, but call it by its name.)
*   Several models per tool: a prediction is a random process (see Lecture 4), a single model tells little.
*   **Write down the settings** you use (tool, version or date, number of models and seeds, templates, MSA) in the report.
*   Start early: the predictions take from minutes to about an hour, free GPUs have quotas, the servers have daily limits and many students use the same MSA servers at the same time.

### ColabFold (AlphaFold2-multimer)

Open https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/AlphaFold2.ipynb , **save a copy** (File menu), choose a GPU runtime (Runtime → Change runtime type) and set:

| parameter | value |
|:--|:--|
| `query_sequence` | the line printed above (the chains are separated by `:`) |
| `jobname` | a short name, without spaces |
| `msa_mode` | `mmseqs2_uniref_env` (default) |
| `model_type` | `auto` (it uses AlphaFold2-multimer for complexes) |
| `num_recycles` | `auto` |
| `template_mode` | `none` |
| `num_relax` | `0` |
| `num_seeds` | `1` (= 5 models; more only if you have time) |

At the end the notebook downloads `<jobname>.result.zip`. **Drag it into the folder `results/colabfold`** of this notebook (File explorer, on the left). Ions cannot be modelled by this tool.

### AlphaFold3 (AlphaFold Server)

Go to https://alphafoldserver.com (a Google account is needed; there is a daily limit of jobs, and the terms of use are for non-commercial use). Add one **entity** for each component (type *protein*, with the number of copies) and one for each ion, as printed above (alternatively you can try to *upload* the json file `inputs/..._af3server.json` written above). Leave the seed on *auto* and run the job: you get 5 models. Download the zip file and **drag it into `results/af3`**.

### A third tool: Boltz

Lecture 4 presents several recent methods (slide 38). We use [Boltz](https://github.com/jwohlwend/boltz), that is open and runs on a Colab GPU. You can run it here (**GPU runtime needed**: Runtime → Change runtime type). Check `!boltz predict --help` if an option has changed.

In [ ]:
run_boltz = False          # set to True to run Boltz in this notebook (needs a GPU runtime)

if run_boltz:
    %pip install -q boltz
    !boltz predict {paths['boltz']} --use_msa_server --diffusion_samples 5 --output_format mmcif --out_dir boltz_out
    import shutil
    shutil.copytree('boltz_out', 'results/boltz', dirs_exist_ok=True)

You can add **other tools** (for example the [Protenix server](https://protenix-server.com/) or the others of the lecture): put the results in a new folder `results/<name of the tool>`. The analysis reads structure files in `.pdb` or `.cif` format; the confidence scores are read automatically for ColabFold, AlphaFold Server and Boltz, for the other tools only the coordinates are used.

## Part 5. Analysis of the models

We read all the models in `results/` (one sub-folder for each tool). If you dragged zip files in the folders, they are unzipped first. Models that do not contain the same chains (for example a SwissModel model that covers only a part of the sequence) are set aside, because they cannot be compared residue by residue.

In [ ]:
unpack_results()
models = load_models('results')
models, set_aside = compatible_models(models)

### Confidence

*   **pLDDT** (0-100) is the confidence of the network in the local structure of each residue: below 50 it is very low (frequently disordered regions), 50-70 low, 70-90 confident, above 90 very high.
*   **PAE** (predicted aligned error, in Å) is the expected error on the position of residue *j* when the model is aligned on residue *i*. The **off-diagonal blocks** tell how confident the network is about the *relative position of the two chains*: dark green (low error) means confident, light means not.

In [ ]:
plot_plddt(models)
plt.show()

best = {}                                   # the best ranked model of each tool (the first one)
for m in models: best.setdefault(m['tool'], m)
with_pae = [m for m in best.values() if m['pae'] is not None]
fig, axes = plt.subplots(1, len(with_pae), figsize=(4.6*len(with_pae), 4), squeeze=False)
for ax, m in zip(axes[0], with_pae): plot_pae(m, ax=ax)
plt.tight_layout()
plt.show()

### Scores of the models

| score | what it is |
|:--|:--|
| **pTM**, **ipTM** (0-1) | confidence of the network on the whole fold / on the *interface* between chains. A rule of thumb often quoted for ipTM is: above 0.8 confident, below 0.6 probably a failed prediction, in between a grey zone. Disordered tails lower these scores, this is why we trim the constructs. |
| **mean pLDDT** | average of the local confidence over all the residues |
| **interface contacts** | pairs of residues of different chains with atoms closer than 5 Å (computed from the coordinates, for all the tools) |
| **interface area** | buried surface area (Å²), $(SASA_A + SASA_B - SASA_{AB})/2$: stable interfaces between globular proteins bury roughly 1000 Å² or more, short peptide interfaces much less |
| **interface pLDDT, interface PAE** | confidence of the network on the residues at the interface, and average PAE between the two chains |
| **pDockQ** | an empirical score (Bryant et al., Nat. Commun. 2022) that estimates the quality of an interface from its pLDDT and number of contacts; it is only a rough guide |

All of them are *estimates of confidence*, not measurements of how correct the model is.

In [ ]:
scores, pairs = summarize(models)
scores.to_csv('scores.csv', index=False)
scores.round(2)

In [ ]:
plot_scores(scores)
plt.show()

### Do the tools agree?

Two ways of comparing two models of a complex, both **independent of the tool**:

*   *shared interface contacts* (Jaccard index, 1 = the same residue pairs are in contact);
*   *L-RMSD*: superimpose the two models on the longest chain and measure the RMSD of the Cα atoms of the other chain(s): it tells how much the partner moved. Residues with a low confidence can be excluded with `PLDDT_MIN` (flexible regions would otherwise dominate the RMSD). If the system contains several identical chains the best matching of the chains is used.

Models are ordered by clustering: **look for blocks**. Do the models of one tool agree with each other? Do different tools agree? A prediction that is reproduced by independent methods is more credible than one that is not (although the tools are trained on similar data, so agreement is not proof).

In [ ]:
PLDDT_MIN = 70           # use only residues with pLDDT above this value in the L-RMSD (None = all)

J, L = compare_models(models, plddt_min=PLDDT_MIN)
order = plot_similarity(J, L, models)
plt.show()
agreement_table(J, L, models).round(2)

### The consensus interface

For every residue: in which fraction of *all* the models is it at the interface? Residues that are at the interface in almost all the models, whatever the tool, are your best candidates for the real interface (and for mutagenesis experiments). Residue numbers are given both for the construct and for the UniProt entry.

In [ ]:
cons = consensus_interface(models, system_chains=chains)
plot_consensus(cons, threshold=0.8)
plt.show()

core = cons[cons['fraction_of_models'] >= 0.8]
print(f'{len(core)} residues are at the interface in at least 80% of the models')
core

Which of these residues are annotated in UniProt, conserved among homologs, or part of a motif that you found in Part 1? Compare with the pLDDT: are they confidently predicted?

### Looking at the models

The models are superposed on the longest chain. Colours: one for each tool (blue ColabFold, red AlphaFold Server, green Boltz, grey others). Rotate with the mouse.

In [ ]:
show_models(models, color='tool').show()

In [ ]:
# the best model of a tool coloured by pLDDT (blue = confident, red = low): choose the tool
tool = list(best)[0]
show_models([best[tool]], color='plddt').show()

Do the components (the individual chains) look the same in all the models? Is the *relative position and orientation* of the partners the same? Where do the differences concentrate?

## Part 6. Controls and hypotheses

Without an experimental structure the confidence scores alone cannot tell if the complex is real. Predictors can produce confident-looking interfaces between proteins that do not interact. **Controls** tell how much the scores mean.

### Negative control: a partner that cannot bind

We replace the second component with a **shuffled** sequence (same length and composition, but no specific interaction) and predict again with the same tool(s) and settings. Be careful when interpreting: a shuffled sequence has no homologs, so the MSA is also very different, this is a *harsh* control. You may also use the sequence of an unrelated protein as partner (`replacement=` in `make_control`).

In [ ]:
CONTROL_COMPONENT = SYSTEM['components'][1]['name']          # the component to replace

ctrl_chains = make_control(chains, CONTROL_COMPONENT, shuffle=True, seed=1)
paths_ctrl = write_inputs(ctrl_chains, SYSTEM.get('ions', []), SYSTEM['name'] + '_shuffled', outdir='inputs')

for tool in ['colabfold', 'af3', 'boltz']:
    os.makedirs(f'results_shuffled/{tool}', exist_ok=True)

Predict the control with at least one tool (the same settings as before), and put the results in `results_shuffled/<tool>`. Then compare the scores of the real system and of the control.

In [ ]:
unpack_results('results_shuffled')
ctrl_models, _ = compatible_models(load_models('results_shuffled'))
ctrl_scores, _ = summarize(ctrl_models)
plot_scores({'real': scores, 'shuffled': ctrl_scores})
plt.show()
ctrl_scores.round(2)

Which scores separate the real system from the control, and by how much? If the control also gets a high score, how much would you trust the same score for your complex?

### Mutations in silico

If your predicted interface is correct, mutating its key residues should weaken the complex. Predictors are often *not* very sensitive to point mutations (they were not trained for it), so this is a test of the prediction and of the predictor at the same time. Choose residues of the consensus interface and mutate them (here an example: the three most confident consensus residues of the first component, mutated to alanine). Predict the mutant and compare it with the wild type.

In [ ]:
first = SYSTEM['components'][0]['name']
first_chain = [c for c in chains if c['component'] == first][0]['id']
top = (core[core['chain'] == first_chain].sort_values('pLDDT_ref', ascending=False).head(3))
display(top)

# {residue number in the construct: new amino acid}, you can edit it
mutations = {int(r.residue): 'A' for r in top.itertuples() if r.name != 'ALA'}
print('mutations:', mutations)

mut_chains = make_control(chains, first, mutations=mutations)
paths_mut = write_inputs(mut_chains, SYSTEM.get('ions', []), SYSTEM['name'] + '_mutant', outdir='inputs')
for tool in ['colabfold', 'af3', 'boltz']:
    os.makedirs(f'results_mutant/{tool}', exist_ok=True)

### Other evidence

Collect evidence **from other sources** and discuss it in the report:

*   **FoldSeek** (https://search.foldseek.com/search): upload the best model of each component (or of the complex) and look at the hits. Are there experimental structures of *homologous complexes*, with a partner similar to yours? Is the interface formed in the same way?
*   **UniProt and literature**: known interactions, mutagenesis, motifs, localisation: is the interaction plausible biologically?
*   **Homology modelling** (Part 3): templates of the complex or of its parts.
*   **The partial answer keys** (the PDB entries listed in the table of Part 1). *Now* that you have your models, download them and compare: does your model reproduce the piece of the assembly that is known experimentally (the E7 LxCxE peptide in the Rb groove, the E7 CR3 dimer, the E2 TAD dimer and the surface it uses to bind the BRD4 C-terminal motif, the fold of the BRCT module, the E6 C-terminal motif in the PDZ groove)? A model that gets a known sub-structure right is more credible on the unknown part, though the predictors may have learned those structures during training, so this is not an independent test. Which part of your complex is *not* covered by any of them? That is your actual prediction.

### Which experiment would test your model?

Propose (in the report) one or two experiments that could confirm or reject your model: for example mutations of the consensus interface residues that should disrupt binding, and compensatory mutations; an assay (pull-down, ITC, crosslinking mass spectrometry...); what result would support the model and what result would reject it.

## Part 7. Save your work for the report

Download the tables and the figures that you want to use. You can save a figure with `plt.savefig('name.png', dpi=150)` before `plt.show()` in the cells above, and then upload it in the report notebook, or recreate the plot there.

In [ ]:
import platform, datetime
scores.to_csv('scores.csv', index=False)
pairs.to_csv('interface_pairs.csv', index=False)
cons.to_csv('consensus_interface.csv', index=False)
print('saved: scores.csv, interface_pairs.csv, consensus_interface.csv')
print('python', platform.python_version(), '| date', datetime.date.today())